# INF-PROF — Working with tabular data using pandas

## Session 2 learning objectives

By the end of this session, you should be able to:

- create and inspect a `DataFrame`;
- import a CSV file;
- select rows and columns with `loc` and `iloc`;
- filter observations with Boolean conditions;
- create or transform columns;
- detect missing values;
- work with dates;
- summarize data with `groupby()`.

The emphasis is on a small, reliable workflow that you can reuse.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("../data")

## 1. Series and DataFrames

A `Series` is one labeled column. A `DataFrame` is a table made of columns.

Columns can have different data types.

In [ ]:
patients = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003"],
    "age": [34, 57, 42],
    "sex": ["F", "M", "F"],
    "weight_kg": [62.0, 91.5, 75.2],
})

patients

## 2. Inspect before transforming

A useful first inspection is:

```python
df.head()
df.shape
df.columns
df.dtypes
df.info()
df.describe()
```

For categorical columns, `value_counts()` is often useful.

In [ ]:
patients.info()
patients.describe()

## 3. Load a real dataset

The repository includes a small synthetic patient dataset so the notebook works
without an Internet connection.

In [ ]:
df = pd.read_csv(DATA / "patients.csv")
df.head()

In [ ]:
df.shape, df.dtypes

## 4. Selecting columns and rows

Prefer explicit column names:

```python
df["age"]
df[["age", "weight_kg"]]
```

Use:

- `.loc[...]` for labels and Boolean conditions;
- `.iloc[...]` for integer positions.

In [ ]:
df.loc[:4, ["patient_id", "age", "group"]]

In [ ]:
df.iloc[:5, :3]

## 5. Filtering observations

Boolean expressions create masks.
Combine conditions with `&` (and) and `|` (or), and put each condition in
parentheses.

In [ ]:
older_treatment = df.loc[
    (df["age"] >= 50) & (df["group"] == "treatment")
]

older_treatment.head()

## 6. Create new columns with vectorized expressions

Pandas usually lets you transform an entire column without an explicit Python loop.

In [ ]:
df["height_m"] = df["height_cm"] / 100
df["bmi"] = df["weight_kg"] / df["height_m"]**2

df[["patient_id", "weight_kg", "height_m", "bmi"]].head()

## 7. Missing values

Missing values are represented by `NaN` / `NA`.

Start by measuring them:

```python
df.isna().sum()
df.isna().mean()
```

Then decide what to do based on the scientific context. Do not automatically
drop or impute values without a reason.

In [ ]:
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "proportion": df.isna().mean(),
})
missing

### Avoid chained `inplace=True`

Code such as:

```python
df["x"].replace(..., inplace=True)
```

is fragile and raises warnings in recent pandas versions.

Prefer assignment:

```python
df["x"] = df["x"].replace(...)
```

In [ ]:
median_weight = df["weight_kg"].median()
df["weight_kg_imputed"] = df["weight_kg"].fillna(median_weight)

## 8. Dates

Convert text to a real datetime type before extracting year, month or weekday.
When a format is known, state it explicitly.

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"], format="%Y-%m-%d")
df["visit_month"] = df["visit_date"].dt.month
df["visit_weekday"] = df["visit_date"].dt.day_name()

df[["visit_date", "visit_month", "visit_weekday"]].head()

## 9. Grouped summaries

The common pattern is:

```python
df.groupby("group").agg(...)
```

Use named aggregations to produce readable output.

In [ ]:
summary = (
    df.groupby("group")
      .agg(
          n=("patient_id", "size"),
          mean_age=("age", "mean"),
          mean_bmi=("bmi", "mean"),
      )
)

summary

## 10. Categorical variables

Do not replace nominal categories by arbitrary numbers unless the model or
analysis genuinely requires an ordering.

For machine learning, one-hot encoding is usually safer for nominal categories.
Pandas can create indicator columns with `pd.get_dummies()`. Later, scikit-learn
pipelines will provide an even safer preprocessing workflow.

In [ ]:
pd.get_dummies(df[["sex", "group"]], dtype=int).head()

## 11. A reproducible pandas pattern

A good data-analysis script usually follows the same sequence:

1. **load** data;
2. **inspect** structure and quality;
3. **clean** known problems;
4. **transform** variables;
5. **summarize** / visualize;
6. keep important decisions explicit in code.

Avoid editing values manually in a spreadsheet when the transformation can be
written in Python.

## Take-home checklist

You should now be comfortable with:

- `read_csv()`;
- `head()`, `info()`, `describe()`;
- `loc`, `iloc`;
- Boolean filtering;
- vectorized column creation;
- `isna()`, `fillna()`;
- `to_datetime()` and `.dt`;
- `groupby()` and `agg()`.